In [1]:
# 1. Setup
import os
from dotenv import load_dotenv
from backend.utils.supabase_client import query_pgvector, client as openai_client

load_dotenv()  # Load environment variables

print("Supabase URL:", os.environ.get("SUPABASE_URL"))
print("OpenAI key loaded?:", bool(os.environ.get("OPENAI_API_KEY")))

# Ensure .env has:
# SUPABASE_URL
# SUPABASE_SERVICE_ROLE_KEY
# OPENAI_API_KEY


Supabase URL: https://zsiaodpoyplzfqommhyf.supabase.co
OpenAI key loaded?: True


In [2]:
# 2. Test the PGVector sub-agent retrieval
question = "What is social anxiety disorder?"
docs = query_pgvector(question, top_k=5)

print("Retrieved chunks:\n")
for i, d in enumerate(docs, 1):
    print(f"{i}. {d['content'][:300]}... (source: {d['source']})")


Retrieved chunks:

1. The National Institute of Mental Health: https://www.nimh.nih.gov/health/publications/social-anxiety-disorder-more-than-just-shyness Social Anxiety Disorder: What You Need to Know En español What is social anxiety disorder? Social anxiety disorder is a common type of anxiety disorder. A person with ... (source: Social Anxiety Disorder_ What You Need to Know - National Institute of Mental Health (NIMH).pdf)
2. end up avoiding places or events that cause distress or generate feelings of embarrassment. In some cases, anxiety may arise only during performance situations such as giving a speech, competing in a sports game, or playing a musical instrument on stage. Social anxiety disorder usually starts during... (source: Social Anxiety Disorder_ What You Need to Know - National Institute of Mental Health (NIMH).pdf)
3. anxiety doesn’t go away and begins to interfere with your daily activities, you may have an anxiety disorder. Learn more about stress. What causes socia

In [3]:
# 3. Ask GPT with context from PGVector
context_text = "\n\n".join([d["content"] for d in docs])

completion = openai_client.chat.completions.create(
    model="gpt-4o-mini",
    messages=[
        {"role": "system", "content": "You are a helpful assistant with access to a knowledge base."},
        {"role": "user", "content": f"Answer the question based on context below.\n\nContext:\n{context_text}\n\nQuestion: {question}"}
    ]
)

print("Answer:\n")
print(completion.choices[0].message.content)


Answer:

Social anxiety disorder is a common type of anxiety disorder characterized by intense anxiety or fear in social situations where an individual may be scrutinized, evaluated, or judged by others. This can occur in scenarios such as public speaking, meeting new people, dating, job interviews, or even everyday activities like eating or drinking in front of others. Individuals with social anxiety disorder often experience symptoms like blushing, sweating, trembling, rapid heart rate, and self-consciousness, and may go to great lengths to avoid these situations. The disorder usually begins in childhood or adolescence, is more prevalent in women than men, and can interfere significantly with daily life. With appropriate treatment and support, those affected can manage their anxiety and improve their quality of life.


In [4]:
# 4. Save outputs (optional)
import os, json
import pandas as pd

os.makedirs("outputs", exist_ok=True)

# Save retrieved docs to CSV
df = pd.DataFrame(docs)
df.to_csv("outputs/subagent_retrieval_test.csv", index=False)

# Save log
with open("outputs/test_log.json", "w") as f:
    json.dump({"question": question, "retrieved": docs}, f, indent=2)

print("Saved results to outputs/")


Saved results to outputs/
